In [79]:
require_relative 'draw_dot'

false

In [74]:
require 'set'

class Value
    attr_reader :data, :children, :op
    attr_accessor :label, :grad, :backward

    def initialize(data, children = [], operator = '', label: '')
        @data = data
        @grad = 0.0
        @backward = -> {}
        @children = children.to_set # consider other data structures?
        @op = operator
        @label = label
    end

    def inspect
        # similar to python just for the lulz
        "Value:(data=#{@data})"
    end

    def +(operand)
        operand = ensure_value(operand)

        out = Value.new( self.data + operand.data, [self, operand], '+')
        out.backward = lambda {
            self.grad += 1.0 * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += 1.0 *out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    
    def *(operand)
        operand = ensure_value(operand)
        
        out = Value.new( self.data * operand.data, [self, operand], '*')
        out.backward = lambda {
            self.grad += operand.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
            operand.grad += self.data * out.grad # * as per chaining rule, += to account for 'a + a' - refer Multivariable case
        }
        out
    end

    # activation (squashing) function
    def tanh
        x = self.data
        t = (Math.exp(2 * x) - 1) / (Math.exp(2 * x) + 1)
        out = Value.new(t, [self], 'tanh - squashing/activation func')

        out.backward = lambda {
            self.grad = (1 - out.data**2) * out.grad
        }
        out
    end

    
    def propagate_all
      topo = []
      visited = Set.new

      build_topo = lambda do |v|
        next if visited.include?(v)

        visited.add(v)
        v.children.each { |child| build_topo.call(child) }
        topo << v
      end

      build_topo.call(self)

      @grad = 1.0
      topo.reverse_each { |v| v.backward.call }
    end

    private

    def ensure_value(operand)
        operand =   
        case operand
        when Value then operand
        when Integer, Float then Value.new(operand)
        else raise("Hey, '#{operand}' must be and Integer or Float")
        end
    end
    
end


:ensure_value

In [75]:
# inputs x
x1 = Value.new(2.0, label: 'axon signal from neuron x1')
x2 = Value.new(0.0, label: 'axon signal from neuron  x2')
# weights w
w1 = Value.new(-3.0, label: 'synapse wight w1')
w2 = Value.new(1.0, label: 'synapse wight w2')
# bias of neuron
b = Value.new(6.8813, label: 'Neuron Bias')

# input signals multiplied by the weights of the dendrite they were received
x1w1 = x1*w1; x1w1.label = 'dendrite signal x1w1'
x2w2 = x2*w2; x2w2.label = 'dendrite signal x2w2'

# then they cumulated
x1w1x2w2 = x1w1 + x2w2; x1w1x2w2.label = 'Sum x1w1x2w2'
# then bias is added to get the BODY OF NEURON, without activation func (output axon)
n = x1w1x2w2 + b; n.label = 'NEURON'
o = n.tanh; o.label = 'output O'

o.propagate_all

auto_graph_o = draw_dot_graphviz(o)
auto_graph_o.output(svg: "neuron.svg")

# Sidequest before neuron (can be skipped)

In [163]:
a = Value.new(2.0)
a + 1

Value:(data=3.0)

In [77]:
# but this fails, unless fixed by below
1 + a

Value:(data=3.0)

In [78]:
Value.class_eval do
    def coerce(other)
      [ensure_value(other), self]
    end
end

:coerce

# Neuron

reminder:

![image](https://www.researchgate.net/publication/364814302/figure/fig5/AS:11431281092677232@1666928276027/Neural-net-Structure-with-an-Activation-Function-Source-CS231n-Stanford-2017.png)


In [155]:
class Neuron
    def initialize(number_of_inputs_for_a_neuron)
        @w = Array.new(number_of_inputs_for_a_neuron) { Value.new(rand(-1.0..1.0), label: 'w') }
        @b = Value.new(rand(-1.0..1.0), label: 'b')
    end

    def call(x)
        nuron_body_for_activation = 
            @w
            .zip(x)
            .map {  |wi, xi|  wi * xi  }
            .sum + @b
            
        nuron_body_for_activation.tanh
    end
end

list_of_signals_x = [2.0, 3.0]
n = Neuron.new(2)
n.call(list_of_signals_x)

Value:(data=-0.9860649154872662)

# Layer of neurons

![MLP](https://www.researchgate.net/publication/334609713/figure/fig1/AS:783455927406593@1563801857102/Multi-Layer-Perceptron-MLP-diagram-with-four-hidden-layers-and-a-collection-of-single.jpg)

In [158]:
class Layer
    def initialize(number_of_inputs_for_a_neuron, number_of_layer_neuron_outputs)
      @neurons = Array.new(number_of_layer_neuron_outputs) do
          Neuron.new(number_of_inputs_for_a_neuron) 
      end
    end

    def call(x)
      out = @neurons.map { |neuron| neuron.call(x) }
      out.length == 1 ? out[0] : out
    end
end

list_of_signals_x = [2.0, 3.0]
n = Layer.new(2, 3) # number_of_inputs_for_a_neuron 2 matches side of list_of_signals_x
n.call(list_of_signals_x)

[Value:(data=-0.4916492709424736), Value:(data=0.9312308860216739), Value:(data=0.6881634796132216)]

# Multi layer perceptron

![impmlp](mlp.jpeg)

In [160]:
class MultiLayerPerceptron
    def initialize(nin, nouts_list)
      sz = [nin] + nouts_list
      @layers = nouts_list.each_index.map do |i|
        Layer.new(sz[i], sz[i + 1])
      end
    end

    def call(x)
      @layers.each { |layer| x = layer.call(x) }
      x
    end
end

list_of_signals_x = [2.0, 3.0, -1.0]

# refer pic above. we build perceptron for hidden and output layers, and call in with a list of signals fron the input layer
# input_layer_size = 3 # og list_of_signals_x.length
# hidden_layer_1_size = 4
# hidden_layer_2_size = 4
# output_layer_size = 1
n = MultiLayerPerceptron.new(3, [4, 4, 1])

n.call(list_of_signals_x)

Value:(data=-0.510146635822128)

In [162]:
n_graph = draw_dot_graphviz(n.call(list_of_signals_x))
n_graph.output(svg: "mlp_graph.svg")